# LeetCode 3986 — Number of Elapsed Seconds Between Two Times bằng Python

Link đề: https://leetcode.com/problems/number-of-elapsed-seconds-between-two-times/ · Độ khó: **Easy** · Chủ đề: Math, String, Simulation

Skeleton LeetCode:

```python
class Solution:
    def secondsBetweenTimes(self, startTime: str, endTime: str) -> int:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm được đối chiếu qua leetcode.doocs.org (Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình. Slug URL được suy ra từ tên bài.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho hai mốc thời gian dạng `"HH:MM:SS"` trong **cùng một ngày** (`endTime` không sớm hơn `startTime`); trả về số **giây** trôi qua.
2. Biết đổi một mốc thời gian thành **số giây kể từ 00:00:00** bằng `h * 3600 + m * 60 + s`, rồi lấy hiệu hai mốc.
3. Biết tách chuỗi bằng `split(":")`, cắt lát theo vị trí cố định, và dùng `map(int, ...)`; làm quen `divmod` để đổi ngược giây sang `HH:MM:SS`.
4. Viết 4 cách (tách `split`, cắt lát theo chỉ số, `datetime`, mô phỏng trừ từng trường có mượn) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh lỗi: cộng giờ/phút/giây không quy đổi, lấy `abs` từng trường, quên `* 60`, nhầm `"HH:MM"` và `"MM:SS"`.

## 2. Đề bài & ví dụ

Cho hai thời điểm hợp lệ `startTime` và `endTime`, mỗi thời điểm là chuỗi dạng `"HH:MM:SS"` (giờ 24h). Hãy trả về **số giây** đã trôi qua từ `startTime` đến `endTime`.

| Ví dụ | `startTime`, `endTime` | Output | Giải thích |
|---|---|---|---|
| 1 | `"01:00:00"`, `"01:00:25"` | `25` | `endTime` muộn hơn 25 giây |
| 2 | `"12:34:56"`, `"13:00:00"` | `1504` | Muộn hơn 25 phút 4 giây `= 25*60 + 4 = 1504` giây |

Constraints: `len(startTime) == len(endTime) == 8`; cả hai là thời điểm hợp lệ dạng `"HH:MM:SS"`; `00 <= HH <= 23`, `00 <= MM <= 59`, `00 <= SS <= 59`; **`endTime` không sớm hơn `startTime`** (cùng ngày, không qua nửa đêm).

Hằng số cần nhớ: 1 phút = 60 giây, 1 giờ = 3600 giây, một ngày có `86400` giây.

In [1]:
for a, b in [("01:00:00", "01:00:25"), ("12:34:56", "13:00:00")]:
    ha, ma, sa = map(int, a.split(":"))
    hb, mb, sb = map(int, b.split(":"))
    ta, tb = ha * 3600 + ma * 60 + sa, hb * 3600 + mb * 60 + sb
    print(f"{a} = {ta} giây, {b} = {tb} giây -> hiệu = {tb - ta}")

01:00:00 = 3600 giây, 01:00:25 = 3625 giây -> hiệu = 25
12:34:56 = 45296 giây, 13:00:00 = 46800 giây -> hiệu = 1504


## 3. Kiến thức nền

### 3.1. Tách chuỗi bằng `split`

`"12:34:56".split(":")` trả về list `['12', '34', '56']`. Các phần tử vẫn là **chuỗi**, cần `int(...)` để tính toán. `int("07")` là `7` (số `0` đứng đầu không gây lỗi).

In [2]:
parts = "12:34:56".split(":")
print(parts, [int(p) for p in parts], int("07"), int("00"))

['12', '34', '56'] [12, 34, 56] 7 0


### 3.2. `map(int, ...)` và gán nhiều biến

`h, m, s = map(int, "12:34:56".split(":"))` gán ba số cùng lúc (unpacking). `map` trả về iterator, `h, m, s = ...` tiêu thụ đúng ba giá trị.

In [3]:
h, m, s = map(int, "12:34:56".split(":"))
print(h, m, s)

12 34 56


### 3.3. Cắt lát theo vị trí cố định

Vì định dạng luôn `"HH:MM:SS"` dài đúng `8` ký tự, giờ là `t[0:2]`, phút là `t[3:5]`, giây là `t[6:8]`. Cách này không cần `split`; hợp khi định dạng chắc chắn cố định.

In [4]:
t = "12:34:56"
print(t[0:2], t[3:5], t[6:8], int(t[0:2]) * 3600 + int(t[3:5]) * 60 + int(t[6:8]))

12 34 56 45296


### 3.4. Quy đổi đơn vị thời gian và `divmod`

`giây = h * 3600 + m * 60 + s`. Chiều ngược lại: `m, s = divmod(giây, 60)`, `h, m = divmod(m, 60)`. Hai chiều là nghịch đảo nhau: dùng để **kiểm tra chéo** kết quả.

In [5]:
total = 45296
m, s = divmod(total, 60)
h, m = divmod(m, 60)
print(h, m, s, f"{h:02d}:{m:02d}:{s:02d}")

12 34 56 12:34:56


### 3.5. Module `datetime` (tham khảo)

`datetime.strptime("12:34:56", "%H:%M:%S")` phân tích chuỗi thành đối tượng thời gian; hiệu hai `datetime` là `timedelta`, có `total_seconds()`. Mạnh nhưng nặng; với định dạng cố định đơn giản, tự quy đổi nhanh và rõ hơn. (Chỉ dùng thư viện chuẩn.)

In [6]:
from datetime import datetime
a = datetime.strptime("12:34:56", "%H:%M:%S")
b = datetime.strptime("13:00:00", "%H:%M:%S")
print(b - a, (b - a).total_seconds())

0:25:04 1504.0


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Đồng hồ là một **số đếm giây** kể từ nửa đêm trình bày dưới dạng `HH:MM:SS`. Nếu đổi **cả hai** mốc về cùng đơn vị (giây) thì "thời gian trôi qua" chỉ là một phép **trừ**.

Tính tay `"12:34:56"` -> `12*3600 + 34*60 + 56 = 43200 + 2040 + 56 = 45296` giây.
Tính tay `"13:00:00"` -> `13*3600 = 46800` giây.
Hiệu `46800 - 45296 = 1504`. Khớp ví dụ 2.

| mốc | giờ | phút | giây | tổng giây |
|---|---|---|---|---|
| `12:34:56` | 12 | 34 | 56 | 45296 |
| `13:00:00` | 13 | 0 | 0 | 46800 |
| `00:00:00` | 0 | 0 | 0 | 0 |
| `23:59:59` | 23 | 59 | 59 | 86399 |

In [7]:
def to_seconds(t):
    h, m, s = map(int, t.split(":"))
    return h * 3600 + m * 60 + s

for t in ("12:34:56", "13:00:00", "00:00:00", "23:59:59"):
    print(t, to_seconds(t))
print(to_seconds("13:00:00") - to_seconds("12:34:56"))

12:34:56 45296
13:00:00 46800
00:00:00 0
23:59:59 86399
1504


## 5. Cách 1 — Brute Force / baseline: đếm từng giây bằng mô phỏng đồng hồ

Cách "ngây thơ": cho đồng hồ chạy từ `startTime`, mỗi lần tăng **1 giây** (có xử lý tràn giây -> phút -> giờ) cho đến khi bằng `endTime`, đếm số lần tăng.

Đúng nhưng chậm: tối đa `86399` lần tăng. Nút thắt: ta đang **đếm** thứ có thể **tính** bằng phép trừ.

In [8]:
class SolutionSimulate:
    def secondsBetweenTimes(self, startTime, endTime):
        h, m, s = map(int, startTime.split(":"))
        eh, em, es = map(int, endTime.split(":"))
        count = 0
        while (h, m, s) != (eh, em, es):
            s += 1
            if s == 60:
                s = 0
                m += 1
                if m == 60:
                    m = 0
                    h += 1
            count += 1
        return count

print(SolutionSimulate().secondsBetweenTimes("01:00:00", "01:00:25"), SolutionSimulate().secondsBetweenTimes("12:34:56", "13:00:00"))

25 1504


**Độ phức tạp:** Time `O(số giây trôi qua) <= O(86400)`, Space `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** đổi mỗi mốc về **giây kể từ 00:00:00** bằng công thức `h * 3600 + m * 60 + s`, rồi lấy **hiệu**. Một phép tính `O(1)` thay cho 86400 lần tăng.

`elapsed = toSeconds(endTime) - toSeconds(startTime)`

**Quy tắc:** vì đề đảm bảo `endTime >= startTime` cùng một ngày nên hiệu **không âm**, không cần xử lý qua nửa đêm.

**Invariant:** hàm `toSeconds` là **đơn điệu tăng** theo thứ tự thời gian: mốc muộn hơn cho số giây lớn hơn (vì `m < 60`, `s < 60`, nên không có "chồng lấn" giữa các trường). Do đó phép trừ phản ánh đúng khoảng cách thời gian.

Cách thứ ba (mượn từng trường) cho cùng kết quả: trừ `giây`, `phút`, `giờ` riêng và quy đổi.

## 7. Các cách tiếp theo

### Cách 2 — Đổi sang giây bằng `split` + `map`

Cách chuẩn, ngắn gọn: viết hàm phụ `to_sec(t)` rồi lấy hiệu.

In [9]:
class SolutionSplit:
    def secondsBetweenTimes(self, startTime, endTime):
        def to_sec(t):
            h, m, s = map(int, t.split(":"))
            return h * 3600 + m * 60 + s
        return to_sec(endTime) - to_sec(startTime)

print(SolutionSplit().secondsBetweenTimes("01:00:00", "01:00:25"), SolutionSplit().secondsBetweenTimes("12:34:56", "13:00:00"))

25 1504


**Độ phức tạp:** Time `O(1)` (chuỗi cố định 8 ký tự), Space `O(1)`.

### Cách 3 — Cắt lát theo vị trí cố định

Vì chuỗi luôn dài 8 ký tự: `int(t[:2])`, `int(t[3:5])`, `int(t[6:])`. Không cần `split`/`map`.

In [10]:
class SolutionSlice:
    def secondsBetweenTimes(self, startTime, endTime):
        def to_sec(t):
            return int(t[:2]) * 3600 + int(t[3:5]) * 60 + int(t[6:])
        return to_sec(endTime) - to_sec(startTime)

print(SolutionSlice().secondsBetweenTimes("01:00:00", "01:00:25"), SolutionSlice().secondsBetweenTimes("12:34:56", "13:00:00"))

25 1504


**Độ phức tạp:** Time `O(1)`, Space `O(1)`.

### Cách 4 — Dùng `datetime`

`datetime.strptime` hiểu định dạng; `(end - start).seconds` hoặc `.total_seconds()`. Dùng `int(...)` để trả về số nguyên. Tiện nếu sau này đề mở rộng (nhiều ngày, múi giờ), nhưng chậm hơn tự quy đổi.

In [11]:
from datetime import datetime

class SolutionDatetime:
    def secondsBetweenTimes(self, startTime, endTime):
        fmt = "%H:%M:%S"
        delta = datetime.strptime(endTime, fmt) - datetime.strptime(startTime, fmt)
        return int(delta.total_seconds())

print(SolutionDatetime().secondsBetweenTimes("01:00:00", "01:00:25"), SolutionDatetime().secondsBetweenTimes("12:34:56", "13:00:00"))

25 1504


**Độ phức tạp:** Time `O(1)`, Space `O(1)` (hằng số lớn hơn do phân tích định dạng).

### Cách 5 — Trừ từng trường rồi quy đổi (không cần hàm `to_sec`)

`(eh - sh) * 3600 + (em - sm) * 60 + (es - ss)`. Các hiệu từng trường có thể **âm** (ví dụ phút `0 - 34`), nhưng tổng quy đổi vẫn đúng vì đây chỉ là phân phối phép trừ.

In [12]:
class SolutionFieldDiff:
    def secondsBetweenTimes(self, startTime, endTime):
        sh, sm, ss = map(int, startTime.split(":"))
        eh, em, es = map(int, endTime.split(":"))
        return (eh - sh) * 3600 + (em - sm) * 60 + (es - ss)

print(SolutionFieldDiff().secondsBetweenTimes("01:00:00", "01:00:25"), SolutionFieldDiff().secondsBetweenTimes("12:34:56", "13:00:00"))

25 1504


**Độ phức tạp:** Time `O(1)`, Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 2 với `startTime = "12:34:56"`, `endTime = "13:00:00"`:

| bước | mốc | `h` | `m` | `s` | `h*3600` | `m*60` | `s` | tổng giây |
|---|---|---|---|---|---|---|---|---|
| start | `12:34:56` | 12 | 34 | 56 | 43200 | 2040 | 56 | 45296 |
| end | `13:00:00` | 13 | 0 | 0 | 46800 | 0 | 0 | 46800 |

Hiệu `46800 - 45296 = 1504`.

In [13]:
def trace_to_sec(label, t):
    h, m, s = map(int, t.split(":"))
    total = h * 3600 + m * 60 + s
    print(f"{label} {t}: {h}*3600 + {m}*60 + {s} = {h * 3600} + {m * 60} + {s} = {total}")
    return total

def trace_elapsed(a, b):
    ta = trace_to_sec("start", a)
    tb = trace_to_sec("end  ", b)
    print(f"elapsed = {tb} - {ta} = {tb - ta}")
    m, s = divmod(tb - ta, 60)
    h, m = divmod(m, 60)
    print(f"  (đổi ngược: {h} giờ {m} phút {s} giây)")
    return tb - ta

assert trace_elapsed("12:34:56", "13:00:00") == 1504
assert trace_elapsed("01:00:00", "01:00:25") == 25
assert trace_elapsed("00:00:00", "23:59:59") == 86399

start 12:34:56: 12*3600 + 34*60 + 56 = 43200 + 2040 + 56 = 45296
end   13:00:00: 13*3600 + 0*60 + 0 = 46800 + 0 + 0 = 46800
elapsed = 46800 - 45296 = 1504
  (đổi ngược: 0 giờ 25 phút 4 giây)
start 01:00:00: 1*3600 + 0*60 + 0 = 3600 + 0 + 0 = 3600
end   01:00:25: 1*3600 + 0*60 + 25 = 3600 + 0 + 25 = 3625
elapsed = 3625 - 3600 = 25
  (đổi ngược: 0 giờ 0 phút 25 giây)
start 00:00:00: 0*3600 + 0*60 + 0 = 0 + 0 + 0 = 0
end   23:59:59: 23*3600 + 59*60 + 59 = 82800 + 3540 + 59 = 86399
elapsed = 86399 - 0 = 86399
  (đổi ngược: 23 giờ 59 phút 59 giây)


In [14]:
def trace_simulate(a, b, limit=8):
    h, m, s = map(int, a.split(":"))
    target = tuple(map(int, b.split(":")))
    count = 0
    print("mô phỏng:", (h, m, s), "->", target)
    while (h, m, s) != target:
        s += 1
        if s == 60:
            s, m = 0, m + 1
            if m == 60:
                m, h = 0, h + 1
        count += 1
        if count <= limit:
            print(f"  +1s -> {h:02d}:{m:02d}:{s:02d}  (đã đếm {count})")
    print("tổng số giây đếm được:", count)
    return count

assert trace_simulate("10:59:57", "11:00:02") == 5

mô phỏng: (10, 59, 57) -> (11, 0, 2)
  +1s -> 10:59:58  (đã đếm 1)
  +1s -> 10:59:59  (đã đếm 2)
  +1s -> 11:00:00  (đã đếm 3)
  +1s -> 11:00:01  (đã đếm 4)
  +1s -> 11:00:02  (đã đếm 5)
tổng số giây đếm được: 5


## 9. Code nộp LeetCode

Dùng Cách 2 (`split` + `map`, ngắn và rõ nghĩa).

In [15]:
from typing import List, Optional


class Solution:
    def secondsBetweenTimes(self, startTime: str, endTime: str) -> int:
        def to_sec(t: str) -> int:
            h, m, s = map(int, t.split(":"))
            return h * 3600 + m * 60 + s

        return to_sec(endTime) - to_sec(startTime)


print(Solution().secondsBetweenTimes("01:00:00", "01:00:25"), Solution().secondsBetweenTimes("12:34:56", "13:00:00"))

25 1504


## 10. Vì sao thuật toán đúng

1. **Mốc thời gian là số giây.** Một thời điểm `h:m:s` (`0 <= h < 24`, `0 <= m, s < 60`) cách `00:00:00` đúng `3600 h + 60 m + s` giây, vì một giờ có 3600 giây, một phút có 60 giây.
2. **Đơn ánh.** Vì `m, s <= 59`, `60 m + s <= 3599 < 3600` nên `3600 h + 60 m + s` xác định duy nhất bộ `(h, m, s)` (chữ số "mượn" không chồng lấn). Mốc muộn hơn có số giây lớn hơn.
3. **Khoảng thời gian = hiệu số giây.** Số giây từ `A` đến `B` bằng `sec(B) - sec(A)` vì cả hai cùng đo từ gốc `00:00:00`.
4. **Không âm.** Đề đảm bảo `endTime` không sớm hơn `startTime` (cùng ngày) nên `sec(end) >= sec(start)`.
5. **Các cách khác.** Cách 3 chỉ đọc `h, m, s` bằng cắt lát; Cách 5 phân phối phép trừ (`(eh - sh) 3600 + (em - sm) 60 + (es - ss) = sec(end) - sec(start)` theo tính phân phối); Cách 4 do thư viện; Cách 1 đếm từng giây nên cho đúng cùng số.

## 11. Phân tích Time / Space Complexity

- Cách 1 (mô phỏng): Time `O(T)` với `T <= 86400` là kết quả; Space `O(1)`.
- Cách 2, 3, 5: Time `O(1)` (độ dài chuỗi cố định 8), Space `O(1)`.
- Cách 4 (`datetime`): Time `O(1)` nhưng hằng số lớn; Space `O(1)`.

Cách nộp `O(1)`: vài phép nhân/cộng. Bài này không có "kích thước đầu vào" thay đổi nên `O(1)` là tối ưu.

## 12. Bộ test

### Cách kiểm thử trong notebook này

Chúng ta dùng ba lớp kiểm thử, từ dễ đến khó:

1. **Test cố định**: ví dụ của đề + các trường hợp biên tự nghĩ ra. Mỗi test là một `assert hàm(input) == kết_quả_mong_đợi`. Nếu sai, Python ném `AssertionError` và dừng ngay, kèm thông tin ta in ra để dễ lần lại.
2. **Test ngẫu nhiên có `random.seed`**: gọi `random.seed(số)` trước vòng lặp để **mỗi lần chạy lại sinh đúng cùng dữ liệu** (tái lập được). Input luôn được sinh **trong đúng constraints** của đề.
3. **Đối chiếu nhiều cách (oracle)**: một cách rất đơn giản, hiển nhiên đúng (brute force) đóng vai "trọng tài"; mọi cách nhanh hơn phải cho cùng kết quả với trọng tài trên mọi input ngẫu nhiên.

In [16]:
import random

def fmt(sec):
    m, s = divmod(sec, 60)
    h, m = divmod(m, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

solutions = [SolutionSplit(), SolutionSlice(), SolutionDatetime(), SolutionFieldDiff(), Solution()]

fixed = [
    ("01:00:00", "01:00:25", 25), ("12:34:56", "13:00:00", 1504), ("00:00:00", "00:00:00", 0),
    ("00:00:00", "23:59:59", 86399), ("23:59:59", "23:59:59", 0), ("00:59:59", "01:00:00", 1),
    ("09:05:07", "10:04:06", 3539), ("11:11:11", "22:22:22", 40271),
    ("00:00:59", "00:01:00", 1), ("05:30:00", "17:45:30", 44130), ("23:00:00", "23:00:01", 1),
]
for a, b, expected in fixed:
    for sol in solutions + [SolutionSimulate()]:
        assert sol.secondsBetweenTimes(a, b) == expected, (type(sol).__name__, a, b)
print("test cố định OK:", len(fixed), "ca")

random.seed(3986)
for _ in range(1000):
    x, y = sorted((random.randint(0, 86399), random.randint(0, 86399)))
    a, b = fmt(x), fmt(y)
    for sol in solutions:
        assert sol.secondsBetweenTimes(a, b) == y - x, (type(sol).__name__, a, b)
for _ in range(10):                         # mô phỏng từng giây chỉ chạy trên khoảng ngắn
    x = random.randint(0, 86000); d = random.randint(0, 300)
    assert SolutionSimulate().secondsBetweenTimes(fmt(x), fmt(x + d)) == d
print("test ngẫu nhiên OK")

test cố định OK: 11 ca
test ngẫu nhiên OK


### Test thuộc tính (property test)

(1) `elapsed(a, a) = 0`. (2) **Cộng được:** nếu `a <= b <= c` thì `elapsed(a, c) = elapsed(a, b) + elapsed(b, c)`. (3) Kết quả luôn nằm trong `[0, 86399]`. (4) `fmt(to_sec(t)) == t` (đổi qua lại hai chiều).

In [17]:
random.seed(39860)
for _ in range(500):
    x, y, z = sorted(random.randint(0, 86399) for _ in range(3))
    a, b, c = fmt(x), fmt(y), fmt(z)
    S = Solution()
    assert S.secondsBetweenTimes(a, a) == 0
    assert S.secondsBetweenTimes(a, c) == S.secondsBetweenTimes(a, b) + S.secondsBetweenTimes(b, c)
    assert 0 <= S.secondsBetweenTimes(a, c) <= 86399
    assert fmt(to_seconds(a)) == a
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: lấy `abs` của hiệu từng trường

Hiệu phút có thể âm (`0 - 34`), và âm đó **cần thiết** để trừ bớt giờ đã cộng dư. Lấy `abs` từng trường cho kết quả quá lớn.

In [18]:
def wrong_abs_fields(a, b):
    sh, sm, ss = map(int, a.split(":"))
    eh, em, es = map(int, b.split(":"))
    return abs(eh - sh) * 3600 + abs(em - sm) * 60 + abs(es - ss)          # SAI

print(wrong_abs_fields("12:34:56", "13:00:00"), "!=", Solution().secondsBetweenTimes("12:34:56", "13:00:00"))
assert wrong_abs_fields("12:34:56", "13:00:00") == 3600 + 34 * 60 + 56 and Solution().secondsBetweenTimes("12:34:56", "13:00:00") == 1504

5696 != 1504


### Lỗi 2: cộng giờ, phút, giây **mà không quy đổi** về cùng đơn vị

`1 + 0 + 25` coi giờ, phút, giây như nhau: sai hoàn toàn.

In [19]:
def wrong_no_convert(a, b):
    sh, sm, ss = map(int, a.split(":"))
    eh, em, es = map(int, b.split(":"))
    return (eh - sh) + (em - sm) + (es - ss)                               # SAI: không nhân 3600 / 60

print(wrong_no_convert("12:34:56", "13:00:00"), "!=", Solution().secondsBetweenTimes("12:34:56", "13:00:00"))
assert wrong_no_convert("12:34:56", "13:00:00") == 1 - 34 - 56 and Solution().secondsBetweenTimes("12:34:56", "13:00:00") == 1504

-89 != 1504


### Lỗi 3: chỉ lấy hiệu của trường giây

Chỉ xét `es - ss` bỏ qua giờ và phút: đúng với ví dụ 1, sai với ví dụ 2 (`0 - 56 = -56`).

In [20]:
def wrong_seconds_only(a, b):
    return int(b[6:]) - int(a[6:])                                          # SAI

print(wrong_seconds_only("01:00:00", "01:00:25"), wrong_seconds_only("12:34:56", "13:00:00"))
assert wrong_seconds_only("01:00:00", "01:00:25") == 25 and wrong_seconds_only("12:34:56", "13:00:00") == -56

25 -56


### Lỗi 4: nhầm hệ số (`* 60` cho giờ, `* 3600` cho phút)

Giờ nhân `3600`, phút nhân `60`. Đổi chỗ hai hệ số cho kết quả vô lý.

In [21]:
def wrong_coeff(t):
    h, m, s = map(int, t.split(":"))
    return h * 60 + m * 3600 + s                                            # SAI: đảo hệ số

print(wrong_coeff("12:34:56"), "!=", to_seconds("12:34:56"))
assert wrong_coeff("12:34:56") != to_seconds("12:34:56") and to_seconds("12:34:56") == 45296

123176 != 45296


### Lỗi 5: trừ hai **chuỗi** (hoặc quên `int`) / so sánh chuỗi để tìm mốc muộn hơn

`"13:00:00" - "12:34:56"` gây `TypeError`. Ngoài ra, so sánh chuỗi `"HH:MM:SS"` theo thứ tự từ điển **cũng đúng** nhờ định dạng cố định (có số `0` đứng đầu) nhưng không dùng để tính khoảng cách.

In [22]:
try:
    "13:00:00" - "12:34:56"
except TypeError as e:
    print("TypeError:", e)
assert "12:34:56" < "13:00:00" and "09:59:59" < "10:00:00"      # thứ tự từ điển khớp thứ tự thời gian
assert Solution().secondsBetweenTimes("09:59:59", "10:00:00") == 1

TypeError: unsupported operand type(s) for -: 'str' and 'str'


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài "khoảng cách giữa hai mốc thời gian/ngày giờ" -> **đổi cả hai về một đơn vị nhỏ nhất** (giây, phút) rồi trừ.
- Hằng số: `1 giờ = 3600 s`, `1 phút = 60 s`, `1 ngày = 86400 s`. Đổi ngược bằng `divmod` hai lần.
- Định dạng cố định -> có thể cắt lát `t[0:2]`, `t[3:5]`, `t[6:8]` hoặc `split(":")`; chọn cách bạn thấy ít lỗi.
- Cẩn thận ràng buộc "không qua nửa đêm"; nếu có thể qua nửa đêm, dùng `(b - a) % 86400`.
- Tự kiểm tra bằng **hai chiều**: `fmt(to_sec(t)) == t`.
- `datetime` dùng được nhưng thường là "đại bác bắn muỗi"; trong thi đấu hãy tự quy đổi.

## 15. Biến thể của bài

1. **Number of Days Between Two Dates** (1360): hiệu hai ngày (cần số ngày kể từ gốc).
2. **Day of the Year** (1154), **Day of the Week** (1185): quy đổi ngày.
3. **Minimum Time Difference** (539): chênh lệch phút nhỏ nhất giữa các mốc `"HH:MM"` trong list (có vòng 24h).
4. **Convert the Time** (2224), **Add Seconds to Time**: cộng thời gian.
5. Biến thể: `endTime` có thể **sớm hơn** `startTime` (qua nửa đêm): `(sec(end) - sec(start)) % 86400`.
6. Biến thể: nhiều ngày (`"DD:HH:MM:SS"`) hoặc có mili-giây.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng từng giây | tăng đồng hồ 1s mỗi lần | `O(T)` | `O(1)` | Chỉ để thấy quy đổi, chậm |
| 2. `split` + `map` | đổi sang giây rồi trừ | `O(1)` | `O(1)` | **Nộp bài** |
| 3. Cắt lát cố định | `t[:2]`, `t[3:5]`, `t[6:]` | `O(1)` | `O(1)` | Định dạng chắc chắn cố định |
| 4. `datetime` | `strptime` + `timedelta` | `O(1)` | `O(1)` | Đề mở rộng (nhiều ngày, múi giờ) |
| 5. Hiệu từng trường | `(dh)*3600 + (dm)*60 + ds` | `O(1)` | `O(1)` | Ít code, không cần hàm phụ |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().secondsBetweenTimes("00:00:00", "00:01:30")` và `Solution().secondsBetweenTimes("10:10:10", "11:11:11")` ra gì?

**Bài 2 (tính tay):** từ `"07:45:30"` đến `"09:15:10"` trôi qua bao nhiêu giây? Đổi kết quả về giờ:phút:giây.

**Bài 3 (điền TODO):** hoàn thành `seconds_wrap(a, b)`: nếu `b` sớm hơn `a` thì coi `b` là ngày hôm sau (qua nửa đêm).

**Bài 4 (tự viết lại):** viết `format_seconds(sec)` trả về `"HH:MM:SS"` (đệm số `0` bên trái bằng `:02d`).

**Bài 5 (mini challenge):** viết `min_gap(times)`: khoảng cách giây nhỏ nhất giữa hai mốc **khác vị trí** trong list (đã cùng một ngày, không vòng).

In [23]:
# Bài 1: đoán rồi chạy
print(Solution().secondsBetweenTimes("00:00:00", "00:01:30"), Solution().secondsBetweenTimes("10:10:10", "11:11:11"))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def seconds_wrap_todo(a, b):
    # TODO: tính hiệu rồi dùng % 86400 để xử lý trường hợp qua nửa đêm
    return 0

90 3661


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: `1 phút 30 giây = 90` giây; `10:10:10 -> 11:11:11` là `1 giờ 1 phút 1 giây`. Bài 2: tính `9:15:10 - 7:45:30` bằng cách đổi cả hai về giây.

**Mức 2 (vừa):** Bài 3: `(to_sec(b) - to_sec(a)) % 86400`. Bài 4: `divmod` hai lần rồi `f"{h:02d}:{m:02d}:{s:02d}"`. Bài 5: sắp xếp danh sách giây rồi xét hiệu các cặp liên tiếp.

**Mức 3 (gần đáp án):** Bài 1: `90` và `3661`. Bài 2: `5380` giây = `01:29:40`. Bài 3: `("23:59:50", "00:00:10") -> 20`. Bài 5: `["01:00:00", "03:00:00", "03:00:05"] -> 5`.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
assert Solution().secondsBetweenTimes("00:00:00", "00:01:30") == 90
assert Solution().secondsBetweenTimes("10:10:10", "11:11:11") == 3661
assert Solution().secondsBetweenTimes("07:45:30", "09:15:10") == 5380 and fmt(5380) == "01:29:40"

# Bài 3
def seconds_wrap(a, b):
    return (to_seconds(b) - to_seconds(a)) % 86400

assert seconds_wrap("23:59:50", "00:00:10") == 20 and seconds_wrap("12:34:56", "13:00:00") == 1504 and seconds_wrap("05:00:00", "05:00:00") == 0

# Bài 4
def format_seconds(sec):
    m, s = divmod(sec, 60)
    h, m = divmod(m, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

assert format_seconds(0) == "00:00:00" and format_seconds(86399) == "23:59:59" and format_seconds(3661) == "01:01:01"

# Bài 5
def min_gap(times):
    secs = sorted(to_seconds(t) for t in times)
    return min(b - a for a, b in zip(secs, secs[1:]))

assert min_gap(["01:00:00", "03:00:00", "03:00:05"]) == 5 and min_gap(["10:00:00", "10:00:00"]) == 0
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

### Template

```python
class Solution:
    def secondsBetweenTimes(self, startTime: str, endTime: str) -> int:
        def to_sec(t: str) -> int:
            h, m, s = map(int, t.split(":"))
            return h * 3600 + m * 60 + s      # giờ*3600 + phút*60 + giây

        return to_sec(endTime) - to_sec(startTime)
```

Khung chung "khoảng cách giữa hai mốc": đổi về **một đơn vị** rồi **trừ**. Nếu có thể qua nửa đêm: `% 86400`.

### Checklist

- [ ] Đã đọc kỹ constraints (độ dài, miền giá trị) và biết thuật toán chạy được trong giới hạn đó.
- [ ] Đã chạy tay ít nhất **một ví dụ của đề** và một **trường hợp biên** (nhỏ nhất, lớn nhất, giá trị đặc biệt).
- [ ] Đúng **tên class / tên hàm / kiểu trả về** như skeleton LeetCode (đúng `bool`, `int`, `list`...).
- [ ] Không dùng `input()`, không `print` thừa trong code nộp.
- [ ] Không làm hỏng dữ liệu đầu vào nếu đề không cho phép (sao chép trước khi sửa).
- [ ] Đã so sánh với một cách brute force trên input ngẫu nhiên nhỏ.
- [ ] Đã dùng `* 3600` cho giờ, `* 60` cho phút; không `abs` từng trường; có `int(...)` cho từng phần.
- [ ] Đã thử `00:00:00 -> 23:59:59`, hai mốc bằng nhau, và hai mốc chỉ khác 1 giây qua ranh giới phút/giờ.